In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 04 模型模块 (core.models)

提供风控建模相关的模型和工具。

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.models import (
    XGBoostRiskModel, LightGBMRiskModel,
    LogisticRegression, ScoreCard
)

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 选择数值特征
numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3', 
                   '轻花老客海纳子分V1', '天创小额网贷分', '衡枢鉴真分老客版']

# 处理缺失值
df_model = df[numeric_features + ['target']].copy()
df_model = df_model.dropna()

X = df_model[numeric_features]
y = df_model['target']

print(f"样本数: {len(df_model):,}")
print(f"特征数: {len(numeric_features)}")
print(f"坏样本率: {y.mean():.2%}")

样本数: 264
特征数: 8
坏样本率: 14.02%


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {len(X_train):,}, 测试集: {len(X_test):,}")

训练集: 184, 测试集: 80


## 1. XGBoost模型

In [4]:
# 训练XGBoost模型
xgb_model = XGBoostRiskModel(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    min_child_weight=50,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# 预测
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# 评估
from hscredit.core.metrics import ks, auc
print(f"XGBoost KS: {ks(y_test, y_prob_xgb):.4f}")
print(f"XGBoost AUC: {auc(y_test, y_prob_xgb):.4f}")

XGBoost KS: 0.1502
XGBoost AUC: 0.5000


## 2. LightGBM模型

In [5]:
# 训练LightGBM模型
lgb_model = LightGBMRiskModel(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.1,
    random_state=42
)

lgb_model.fit(X_train, y_train)

# 预测
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

print(f"LightGBM KS: {ks(y_test, y_prob_lgb):.4f}")
print(f"LightGBM AUC: {auc(y_test, y_prob_lgb):.4f}")

LightGBM KS: 0.1594
LightGBM AUC: 0.5323


## 3. 逻辑回归模型

In [6]:
# 训练逻辑回归
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

# 预测
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print(f"LR KS: {ks(y_test, y_prob_lr):.4f}")
print(f"LR AUC: {auc(y_test, y_prob_lr):.4f}")

# 查看系数
coef_df = pd.DataFrame({
    '特征': X.columns,
    '系数': lr_model.coef_[0]
}).sort_values('系数', key=abs, ascending=False)
display(coef_df)

LR KS: 0.2622
LR AUC: 0.6324


,特征,系数
5,轻花老客海纳子分V1,0.4329
2,极光欺诈分6v1,0.4295
7,衡枢鉴真分老客版,0.0982
3,青云24,-0.0081
0,中智小牛分C3,-0.0059
6,天创小额网贷分,0.0042
1,珊瑚92,-0.0029
4,占信V3,0.0014


## 4. 评分卡模型

In [7]:
from hscredit.core.binning import OptimalBinning

# 对特征进行分箱
binner = OptimalBinning(method='best_iv', max_n_bins=5)
binner.fit(X_train, y_train)

X_train_binned = binner.transform(X_train)
X_test_binned = binner.transform(X_test)

# 创建评分卡
scorecard = ScoreCard(base_score=600, pdo=20)
scorecard.fit(X_train_binned, y_train)

# 预测分数
scores = scorecard.predict_score(X_test_binned)

print(f"评分范围: {scores.min():.0f} - {scores.max():.0f}")
print(f"平均分: {scores.mean():.0f}")

评分范围: 702 - 860
平均分: 776


## 5. 模型对比

In [8]:
from hscredit.core.metrics import lift_at

models_comparison = pd.DataFrame({
    '模型': ['XGBoost', 'LightGBM', 'LogisticRegression'],
    'KS': [
        ks(y_test, y_prob_xgb),
        ks(y_test, y_prob_lgb),
        ks(y_test, y_prob_lr)
    ],
    'AUC': [
        auc(y_test, y_prob_xgb),
        auc(y_test, y_prob_lgb),
        auc(y_test, y_prob_lr)
    ]
})

display(models_comparison.round(4))

,模型,KS,AUC
0,XGBoost,0.1502,0.5000
1,LightGBM,0.1594,0.5323
2,LogisticRegression,0.2622,0.6324
